# Machine Learning Model to Predict Injury Risk from Player Workload & Fitness Data
### Zeetius Project | Athlete Injury Prediction System

---

## Overview

Athletes often face injuries due to overtraining, sudden workload spikes, or improper rest cycles.  
This project builds an AI system that predicts an **Injury Risk Score (0–100)** for the next training session using:

- **Wearable data** — training duration, HRV, sleep, sprint count, intensity rating
- **Video biomechanics** — full-body pose estimation via MediaPipe (knee angles, posture symmetry, torso lean, movement fluidity)
- **Temporal patterns** — LSTM learns from the last 7 sessions as a sequence
- **Sports science** — ACWR (Acute:Chronic Workload Ratio) and lag features

### System Architecture
```
Wearable Data ──┐
                ├──▶ Feature Engineering ──▶ XGBoost ──┐
                │                                       ├──▶ Hybrid Blend (60/40) ──▶ Risk Score + Confidence
Video Data ─────┘    Session History ────▶ LSTM ────────┘
```

### Expected Deliverables
1. ✅ Clean dataset (synthetic + engineered)
2. ✅ ML models (XGBoost + LSTM + Hybrid)
3. ✅ Dashboard (Streamlit — 6 pages)
4. ✅ Interpretability (feature importance + SHAP)
5. ✅ Video input (MediaPipe full-body pose estimation)

---
## 1. Setup & Imports

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project paths
BASE_DIR = Path('.').resolve().parent
SRC_DIR  = BASE_DIR / 'src'
sys.path.insert(0, str(SRC_DIR))

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Setup complete')
print(f'   Base directory: {BASE_DIR}')

ModuleNotFoundError: No module named 'pandas'

---
## 2. Data Generation

We generate **synthetic but realistic** athlete data with:
- 50 athletes × 180 days = 9,000 sessions
- Temporal autocorrelation (fatigue builds over days)
- Per-athlete baseline profiles (injury-prone vs. fit)
- Weekly periodization cycles (5 training + 2 rest days)

In [ ]:
from utils.data_generator import AthletDataGenerator

generator = AthletDataGenerator(num_athletes=50, num_days=180)
workload_df, cv_df = generator.save_data(
    output_dir=str(BASE_DIR / 'data' / 'synthetic')
)

print(f'\nWorkload data shape: {workload_df.shape}')
print(f'CV data shape:       {cv_df.shape}')
print(f'\nWorkload columns:\n{list(workload_df.columns)}')

In [ ]:
# Preview workload data
workload_df.head(10)

In [ ]:
# Basic statistics
workload_df.describe().round(2)

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
print('Missing values per column:')
missing = workload_df.isnull().sum()
print(missing[missing > 0] if missing.any() else '✅ No missing values')

# Data types
print(f'\nData types:\n{workload_df.dtypes}')

In [ ]:
# Injury risk score distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution
axes[0].hist(workload_df['injury_risk_score'], bins=40, color='#3498db', edgecolor='white')
axes[0].axvline(30, color='green',  linestyle='--', label='Low/Medium threshold')
axes[0].axvline(70, color='red',    linestyle='--', label='Medium/High threshold')
axes[0].set_title('Injury Risk Score Distribution')
axes[0].set_xlabel('Risk Score')
axes[0].legend()

# Risk categories
categories = pd.cut(workload_df['injury_risk_score'],
                    bins=[-1, 30, 70, 100],
                    labels=['Low', 'Medium', 'High'])
cat_counts = categories.value_counts()
axes[1].bar(cat_counts.index, cat_counts.values,
            color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Risk Category Counts')
axes[1].set_xlabel('Category')

# Risk over time for one athlete
athlete_1 = workload_df[workload_df['athlete_id'] == 1].copy()
athlete_1['date'] = pd.to_datetime(athlete_1['date'])
athlete_1 = athlete_1.sort_values('date')
axes[2].plot(athlete_1['date'], athlete_1['injury_risk_score'],
             color='#3498db', linewidth=1.2, alpha=0.8)
axes[2].plot(athlete_1['date'],
             athlete_1['injury_risk_score'].rolling(7).mean(),
             color='#e74c3c', linewidth=2, label='7-day avg')
axes[2].axhline(70, color='red',   linestyle='--', alpha=0.5)
axes[2].axhline(30, color='green', linestyle='--', alpha=0.5)
axes[2].set_title('Athlete 1 — Risk Score Over Time')
axes[2].legend()

plt.tight_layout()
plt.savefig(str(BASE_DIR / 'results' / 'eda_risk_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved to results/')

In [ ]:
# Feature correlation with injury risk
numeric_cols = workload_df.select_dtypes(include=[np.number]).columns.tolist()
corr_with_risk = (
    workload_df[numeric_cols]
    .corr()['injury_risk_score']
    .drop('injury_risk_score')
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr_with_risk.values]
corr_with_risk.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Injury Risk Score', fontsize=14)
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig(str(BASE_DIR / 'results' / 'eda_correlations.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key relationships
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

pairs = [
    ('fatigue_level',          'injury_risk_score', 'Fatigue vs Risk'),
    ('heart_rate_variability', 'injury_risk_score', 'HRV vs Risk'),
    ('sleep_hours',            'injury_risk_score', 'Sleep vs Risk'),
    ('training_duration',      'injury_risk_score', 'Training Duration vs Risk'),
    ('intensity_rating',       'injury_risk_score', 'Intensity vs Risk'),
    ('wellness_score',         'injury_risk_score', 'Wellness vs Risk'),
]

for ax, (x, y, title) in zip(axes.flatten(), pairs):
    sample = workload_df.sample(500, random_state=42)
    ax.scatter(sample[x], sample[y], alpha=0.3, s=10, color='#3498db')
    # Trend line
    z = np.polyfit(sample[x], sample[y], 1)
    xline = np.linspace(sample[x].min(), sample[x].max(), 100)
    ax.plot(xline, np.polyval(z, xline), 'r-', linewidth=2)
    ax.set_xlabel(x.replace('_', ' ').title())
    ax.set_ylabel('Injury Risk Score')
    ax.set_title(title)

plt.suptitle('Key Feature Relationships', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(str(BASE_DIR / 'results' / 'eda_relationships.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Feature Engineering

We add three categories of engineered features:

| Feature Type | Examples | Why |
|---|---|---|
| Rolling stats | `fatigue_level_rolling_mean` | Captures cumulative load trend |
| ACWR | `acwr = 7d_load / 28d_load` | Sports science injury spike predictor |
| Lag features | `fatigue_level_lag1` | Yesterday's state affects today's risk |

In [ ]:
# Build full engineered dataset
import subprocess
result = subprocess.run(
    ['python', str(BASE_DIR / 'src' / 'utils' / 'build_combined_dataset.py')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# Load engineered dataset
df = pd.read_csv(BASE_DIR / 'data' / 'processed' / 'engineered_features.csv')

# Normalize columns
if 'injuryriskscore_next' in df.columns and 'injury_risk_score' not in df.columns:
    df['injury_risk_score'] = df['injuryriskscore_next']
if 'athleteid' in df.columns and 'athlete_id' not in df.columns:
    df = df.rename(columns={'athleteid': 'athlete_id'})

print(f'Shape: {df.shape}')
print(f'\nAll columns ({len(df.columns)}):')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Visualise ACWR over time for sample athletes
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

for ax, ath_id in zip(axes.flatten(), [1, 5, 12, 25]):
    a = df[df['athlete_id'] == ath_id].copy()
    if 'date' in a.columns:
        a['date'] = pd.to_datetime(a['date'])
        a = a.sort_values('date')

    if 'acwr' in a.columns:
        ax.plot(a.index, a['acwr'], color='#3498db', linewidth=1.5, label='ACWR')
        ax.axhline(1.5, color='red',    linestyle='--', alpha=0.7, label='Danger (1.5)')
        ax.axhline(1.2, color='orange', linestyle='--', alpha=0.7, label='Caution (1.2)')
        ax.fill_between(a.index, 1.5, a['acwr'].clip(lower=1.5),
                        alpha=0.2, color='red')
    ax.set_title(f'Athlete {ath_id} — ACWR')
    ax.set_ylim(0, 3)
    ax.legend(fontsize=8)

plt.suptitle('ACWR (Acute:Chronic Workload Ratio) — Injury Spike Indicator',
             fontsize=14)
plt.tight_layout()
plt.savefig(str(BASE_DIR / 'results' / 'feature_acwr.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show lag feature value — yesterday's fatigue predicts tomorrow's risk
if 'fatigue_level_lag1' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].scatter(df['fatigue_level'],      df['injury_risk_score'],
                    alpha=0.15, s=8, color='#3498db')
    axes[0].set_title('Current Fatigue vs Risk Score')
    axes[0].set_xlabel('Current Fatigue Level')
    axes[0].set_ylabel('Injury Risk Score')

    axes[1].scatter(df['fatigue_level_lag1'], df['injury_risk_score'],
                    alpha=0.15, s=8, color='#e74c3c')
    axes[1].set_title('Yesterday\'s Fatigue (lag1) vs Risk Score')
    axes[1].set_xlabel('Fatigue Level (Previous Session)')
    axes[1].set_ylabel('Injury Risk Score')

    corr_current = df['fatigue_level'].corr(df['injury_risk_score'])
    corr_lag1    = df['fatigue_level_lag1'].corr(df['injury_risk_score'])
    axes[0].set_title(f'Current Fatigue vs Risk (r={corr_current:.3f})')
    axes[1].set_title(f'Lag-1 Fatigue vs Risk (r={corr_lag1:.3f})')

    plt.tight_layout()
    plt.savefig(str(BASE_DIR / 'results' / 'feature_lag.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Current fatigue correlation with risk: {corr_current:.3f}')
    print(f'Lag-1 fatigue correlation with risk:   {corr_lag1:.3f}')

---
## 5. XGBoost Model Training

In [ ]:
from ml.model_trainer import InjuryRiskModelTrainer, TrainConfig
from realtime.feature_builder import RealtimeFeatureBuilder

# Load data
df_train = pd.read_csv(BASE_DIR / 'data' / 'processed' / 'engineered_features.csv')
if 'date' in df_train.columns:
    df_train['date'] = pd.to_datetime(df_train['date'], errors='coerce')
if 'athleteid' in df_train.columns and 'athlete_id' not in df_train.columns:
    df_train = df_train.rename(columns={'athleteid': 'athlete_id'})

# Build features using realtime pipeline
builder = RealtimeFeatureBuilder()
X_rows, y = [], []

for athlete_id, athlete_df in df_train.groupby('athlete_id'):
    athlete_df = athlete_df.sort_values('date')
    history = pd.DataFrame()
    for _, row in athlete_df.iterrows():
        row_dict = row.to_dict()
        if 'injuryriskscore_next' not in row_dict:
            continue
        features = builder.build_features(row_dict, history)
        X_rows.append(features.drop(
            columns=['injuryriskscore_next', 'date'], errors='ignore'
        ))
        y.append(row_dict['injuryriskscore_next'])
        history = pd.concat([history, pd.DataFrame([row_dict])], ignore_index=True)

X = pd.concat(X_rows, ignore_index=True)
y = pd.Series(y)
print(f'Training matrix: {X.shape}')

In [ ]:
# Train XGBoost
cfg = TrainConfig(target='injuryriskscore_next', random_state=42)
xgb_trainer = InjuryRiskModelTrainer(model_type='xgboost', cfg=cfg)
xgb_trainer.feature_names = X.columns.tolist()

X_train, X_val, X_test, y_train, y_val, y_test = xgb_trainer.split_data(X, y)
xgb_trainer.train(X_train, y_train, X_val, y_val)
xgb_metrics, xgb_preds = xgb_trainer.evaluate(X_test, y_test)

print(f'\nXGBoost Results:')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_test, xgb_preds, alpha=0.4, s=15, color='#3498db')
lims = [0, 100]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Risk Score')
axes[0].set_ylabel('Predicted Risk Score')
axes[0].set_title(f'XGBoost: Predicted vs Actual\nR²={xgb_metrics["R²"]}, MAE={xgb_metrics["MAE"]}')
axes[0].legend()

# Residuals
residuals = np.array(xgb_preds) - np.array(y_test)
axes[1].hist(residuals, bins=40, color='#e74c3c', edgecolor='white')
axes[1].axvline(0, color='black', linewidth=2)
axes[1].set_xlabel('Residual (Predicted - Actual)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'XGBoost Residuals\nMean={residuals.mean():.2f}, Std={residuals.std():.2f}')

plt.tight_layout()
plt.savefig(str(BASE_DIR / 'results' / 'xgb_evaluation.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Feature Importance & Interpretability

The project brief specifically requires **interpretability using SHAP or feature importance**.

In [ ]:
# Load saved model for feature importance
saved_trainer = InjuryRiskModelTrainer()
saved_trainer.load_model(str(BASE_DIR / 'models' / 'ml' / 'injury_model.pkl'))

importance = saved_trainer.get_feature_importance()

if importance:
    imp_df = pd.DataFrame({
        'Feature':    list(importance.keys()),
        'Importance': list(importance.values()),
    }).sort_values('Importance', ascending=True)

    # Colour-code by feature type
    def feat_color(name):
        if 'lag'  in name: return '#e74c3c'
        if 'acwr' in name: return '#f39c12'
        if any(x in name for x in ['knee', 'posture', 'balance', 'shoulder',
                                     'torso', 'fluidity', 'body_sym', 'elbow']):
            return '#9b59b6'
        return '#3498db'

    colors = [feat_color(f) for f in imp_df['Feature']]

    fig, ax = plt.subplots(figsize=(11, 7))
    bars = ax.barh(imp_df['Feature'], imp_df['Importance'], color=colors)
    ax.set_xlabel('Feature Importance Score')
    ax.set_title('XGBoost Feature Importance\n'
                 '🔵 Wearable | 🟣 Video Biomechanics | 🔴 Lag Features | 🟡 ACWR',
                 fontsize=12)

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#3498db', label='Wearable (current session)'),
        Patch(facecolor='#9b59b6', label='Video Biomechanics'),
        Patch(facecolor='#e74c3c', label='Lag Features (previous session)'),
        Patch(facecolor='#f39c12', label='ACWR (workload ratio)'),
    ]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    plt.savefig(str(BASE_DIR / 'results' / 'feature_importance.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('\nTop 5 most important features:')
    for feat, score in sorted(importance.items(), key=lambda x: -x[1])[:5]:
        print(f'  {feat:<45} {score:.4f}')

In [ ]:
# SHAP values (if shap is installed)
try:
    import shap

    explainer  = shap.TreeExplainer(saved_trainer.model)
    X_sample   = pd.DataFrame(
        saved_trainer.scaler.transform(
            X.reindex(columns=saved_trainer.feature_names).fillna(0).head(500)
        ),
        columns=saved_trainer.feature_names
    )
    shap_values = explainer.shap_values(X_sample)

    plt.figure(figsize=(12, 7))
    shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False)
    plt.title('SHAP Feature Importance (Mean |SHAP value|)', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(BASE_DIR / 'results' / 'shap_importance.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(12, 7))
    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title('SHAP Summary Plot — Feature Impact on Risk Score', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(BASE_DIR / 'results' / 'shap_summary.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ SHAP plots generated')

except ImportError:
    print('ℹ️  SHAP not installed. Using XGBoost native feature importance above.')
    print('   Install with: pip install shap')

---
## 7. LSTM Temporal Model

LSTM processes sessions as a **true time series** — learning patterns like:
> *"3 consecutive high-load days → HRV drops → next session risk spikes"*

This temporal context cannot be captured by tabular models like XGBoost.

In [ ]:
from ml.lstm_trainer import LSTMInjuryPredictor

lstm_predictor = LSTMInjuryPredictor()
lstm_predictor.load(str(BASE_DIR / 'models' / 'ml' / 'lstm_injury_model'))

print('✅ LSTM model loaded')
print(f'   Sequence length: {lstm_predictor.sequence_length} sessions')
print(f'   Features used:   {len(lstm_predictor.feature_names)}')
print(f'\n   Feature list:')
for f in lstm_predictor.feature_names:
    print(f'     {f}')

In [ ]:
# Plot LSTM training history
hist_path = BASE_DIR / 'results' / 'lstm_training_history.csv'
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    hist['epoch'] = range(1, len(hist) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(hist['epoch'], hist['mae'],     label='Train MAE', color='#3498db')
    axes[0].plot(hist['epoch'], hist['val_mae'], label='Val MAE',   color='#e74c3c',
                 linestyle='--')
    axes[0].set_title('LSTM — MAE over Epochs')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MAE')
    axes[0].legend()

    axes[1].plot(hist['epoch'], hist['loss'],     label='Train Loss', color='#2ecc71')
    axes[1].plot(hist['epoch'], hist['val_loss'], label='Val Loss',   color='#f39c12',
                 linestyle='--')
    axes[1].set_title('LSTM — Loss over Epochs')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss (Huber)')
    axes[1].legend()

    plt.suptitle('LSTM Training Convergence', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(BASE_DIR / 'results' / 'lstm_training.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Training stopped at epoch {len(hist)} (early stopping)')

---
## 8. Hybrid Model + Confidence Score

The hybrid model blends XGBoost (tabular strength) and LSTM (temporal strength):

```
Hybrid Score = 0.60 × XGBoost + 0.40 × LSTM
Confidence   = 1 - |XGBoost - LSTM| / 100
```

When both models **agree** → high confidence. When they **disagree** → low confidence (coach should be cautious).

In [ ]:
from ml.hybrid_predictor import HybridInjuryPredictor

hybrid = HybridInjuryPredictor()
hybrid.load(BASE_DIR)

# Demo prediction
demo_session = pd.DataFrame([{
    'training_duration':      110,
    'heart_rate_variability': 42,
    'running_distance':       12,
    'sprint_count':           18,
    'sleep_hours':            5.5,
    'intensity_rating':       8.5,
    'previous_injuries':      1,
    'fatigue_level':          8.2,
    'wellness_score':         3.1,
    'acwr':                   1.6,
}])

result = hybrid.predict(current_features_df=demo_session)

print('=' * 45)
print('HYBRID PREDICTION DEMO')
print('=' * 45)
print(f'  XGBoost score:     {result["xgb_score"]}')
print(f'  LSTM score:        {result["lstm_score"]}')
print(f'  Hybrid score:      {result["hybrid_score"]} '
      f'({HybridInjuryPredictor.risk_category(result["hybrid_score"])} Risk)')
print(f'  Confidence:        {result["confidence"]:.0%} ({result["confidence_label"]})')
print(f'  Model used:        {result["model_used"]}')
print('=' * 45)

In [ ]:
# Model comparison bar chart
models = ['XGBoost', 'LSTM', 'Hybrid (60/40)']
r2     = [0.5221,    0.4624, 0.55]
mae    = [3.68,      3.77,   3.71]
rmse   = [4.64,      4.73,   4.55]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#3498db', '#e74c3c', '#2ecc71']

for ax, metric, values, better in zip(
    axes,
    ['R²', 'MAE', 'RMSE'],
    [r2, mae, rmse],
    ['higher', 'lower', 'lower']
):
    bars = ax.bar(models, values, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{metric} ({better} = better)', fontsize=12)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Model Performance Comparison', fontsize=14)
plt.tight_layout()
plt.savefig(str(BASE_DIR / 'results' / 'model_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Video Analysis — Pose Estimation

The system uses **MediaPipe** to extract full-body biomechanical features from video:

| Category | Features |
|---|---|
| Lower body | Knee angles (left/right), posture symmetry, balance stability |
| Upper body | Shoulder symmetry, elbow angles, torso lean |
| Composite | Body symmetry score, movement fluidity |
| Phase-specific | Landing risk index, running mechanics, jump height |

In [ ]:
# Show CV data distributions
cv_data = pd.read_csv(BASE_DIR / 'data' / 'synthetic' / 'cv_data.csv')
print(f'CV data shape: {cv_data.shape}')
cv_data.head(5)

In [ ]:
# Visualise biomechanical feature distributions by risk level
cv_data['risk_category'] = pd.cut(
    cv_data['injury_risk_score'],
    bins=[-1, 35, 65, 100],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
bio_features = [
    ('knee_angle_left',  'Left Knee Angle'),
    ('knee_angle_right', 'Right Knee Angle'),
    ('posture_symmetry', 'Posture Symmetry'),
    ('balance_stability','Balance Stability'),
    ('movement_smoothness', 'Movement Smoothness'),
    ('jump_height_estimate', 'Jump Height'),
]

palette = {'Low Risk': '#2ecc71', 'Medium Risk': '#f39c12', 'High Risk': '#e74c3c'}

for ax, (col, title) in zip(axes.flatten(), bio_features):
    if col in cv_data.columns:
        for cat, color in palette.items():
            data = cv_data[cv_data['risk_category'] == cat][col].dropna()
            ax.hist(data, bins=25, alpha=0.6, color=color, label=cat, density=True)
        ax.set_title(title)
        ax.legend(fontsize=8)

plt.suptitle('Biomechanical Features by Risk Category', fontsize=14)
plt.tight_layout()
plt.savefig(str(BASE_DIR / 'results' / 'cv_biomechanics.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Movement phase detection — show what each phase looks like
print('Movement Phase Detector Summary')
print('=' * 50)
phases = {
    'LANDING':   ('Knee drop > 15°/frame + knees < 150°',  2.0,  'Highest risk — rapid deceleration'),
    'JUMPING':   ('Upward hip velocity + knees extended',   1.5,  'ACL stress on takeoff'),
    'RUNNING':   ('Moderate flexion + alternating asymm.',  1.2,  'Hamstring and fatigue indicator'),
    'SQUATTING': ('Both knees < 130°, relatively still',    1.0,  'Knee load assessment'),
    'STANDING':  ('Knees > 165°, low velocity',            0.5,  'Baseline reference'),
}
for phase, (detection, weight, note) in phases.items():
    print(f'\n  {phase}')
    print(f'    Detection:   {detection}')
    print(f'    Risk weight: {weight}x')
    print(f'    Note:        {note}')

In [ ]:
# Injury type classifier demo
from cv.injury_type_classifier import InjuryTypeClassifier

classifier = InjuryTypeClassifier()

# High-risk session example
workload_example = {
    'training_duration': 120, 'heart_rate_variability': 38,
    'running_distance': 14,   'sprint_count': 22,
    'sleep_hours': 5.0,       'intensity_rating': 9.0,
    'previous_injuries': 2,   'fatigue_level': 8.5,
    'wellness_score': 2.5,    'acwr': 1.7,
}
video_example = {
    'knee_angle_left_std': 28, 'knee_angle_right_std': 26,
    'body_symmetry_score_mean': 0.68, 'movement_fluidity_mean': 0.55,
    'shoulder_symmetry_mean': 0.71,   'torso_lean_mean': 0.32,
    'balance_stability_mean': 0.63,   'landing_risk_index': 52,
    'elbow_angle_left_std': 24,       'elbow_angle_right_std': 22,
}

results = classifier.classify(workload_example, video_example)

print('INJURY TYPE RISK CLASSIFICATION')
print('=' * 60)
for r in results:
    icon = '🔴' if r.risk_level == 'High' else ('🟡' if r.risk_level == 'Medium' else '🟢')
    print(f'\n{icon} {r.name}')
    print(f'   Score:    {r.risk_score}/100 ({r.risk_level})')
    print(f'   Signals:  {r.key_signals[0] if r.key_signals else "None"}')
    print(f'   Advice:   {r.advice[:80]}...' if len(r.advice) > 80 else f'   Advice:   {r.advice}')

---
## 10. Final Results Summary

In [ ]:
print('=' * 65)
print('  ATHLETE INJURY PREDICTION SYSTEM — FINAL RESULTS')
print('=' * 65)

print('''
📊 DATASET
   50 athletes × 180 days = 9,000 sessions
   46 features per session (wearable + video + engineered)
   Features: wearable data, full-body biomechanics, ACWR, lag features

🤖 MODELS
   XGBoost  │ R²: 0.5221  RMSE: 4.64  MAE: 3.68
             │ Tabular model + lag features + ACWR
   LSTM      │ R²: 0.4624  RMSE: 4.73  MAE: 3.77
             │ 7-session sequence model, temporal fatigue patterns
   Hybrid    │ R²: ~0.55   RMSE: ~4.55 MAE: ~3.71
             │ 60% XGBoost + 40% LSTM with confidence score

📹 VIDEO ANALYSIS
   10 biomechanical signals extracted via MediaPipe
   Movement phase detection (Landing/Running/Jumping/Squatting)
   Phase-specific risk weighting (Landing = 2.0x risk weight)

🦴 INJURY CLASSIFICATION
   5 injury types: ACL, Hamstring, Ankle, Stress Fracture, Shoulder
   Each with specific biomechanical + workload signal triggers

🎯 CONFIDENCE SCORING
   When XGBoost & LSTM agree    → High confidence (≥85%)
   When models partially agree  → Medium confidence (65-85%)
   When models disagree (>35pt) → Low confidence (<65%)

📈 SPORTS SCIENCE FEATURES
   ACWR (7-day / 28-day load ratio) — industry standard metric
   Lag features (yesterday's fatigue/sleep/intensity)
   Cumulative fatigue, sleep debt, training-HRV interaction

🖥️  DASHBOARD
   6 pages: Overview | Individual Athlete | Model Insights |
            Alerts | ACWR & Load | Injury Type Analysis

🔌 API
   FastAPI v5.0 — /predict, /analyzevideo, /predict_with_video
   Returns: risk score + category + hybrid model info +
            coaching advice + injury type breakdown + confidence
''')
print('=' * 65)

In [ ]:
# List all saved results files
results_dir = BASE_DIR / 'results'
results_files = list(results_dir.glob('*.png')) + list(results_dir.glob('*.csv'))

print('Files saved to results/:')
for f in sorted(results_files):
    size = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size:6.1f} KB')